In [6]:
import os
import subprocess


os.system("pip install -q dv-processing 2>/dev/null || true")
os.system("pip install -q importlib-metadata 2>/dev/null || true")

print("Dependencies installed. ")

Dependencies installed. 


In [7]:
import os
import subprocess
import shutil

# ============================================================
# CONFIGURE THESE
# ============================================================

# Path to your video (download from Allsky7.net or local)
VIDEO_FILE = "/kaggle/input/datasets/virajsaws/meteor-dataset-allsky7/2026_05_15_01_29_01_000_012661_trim300_wm.mp4"  # CHANGE THIS

# Output directory for v2e results
V2E_OUTPUT_DIR = "/kaggle/working/v2e_output"

# ============================================================
# INSTALL V2E (one-time)
# ============================================================

print("Installing v2e...")
if not os.path.exists("/kaggle/working/v2e"):
    os.system("git clone -q https://github.com/SensorsINI/v2e.git /kaggle/working/v2e")

os.chdir("/kaggle/working/v2e")

# Install dependencies
os.system("pip install -q -r requirements.txt 2>/dev/null || true")
os.system("pip install -q argcomplete engineering_notation numba scipy opencv-python-headless matplotlib h5py imageio imageio-ffmpeg pyyaml tqdm pandas pydantic pyzmq dill vidgear screeninfo 2>/dev/null || true")

# Create headless easygui stub (Kaggle has no GUI)
easygui_stub = "/kaggle/working/v2e/easygui.py"
with open(easygui_stub, "w") as f:
    f.write("def fileopenbox(*args, **kwargs):\n    return None\n")
    f.write("def diropenbox(*args, **kwargs):\n    return None\n")
    f.write("def filesavebox(*args, **kwargs):\n    return None\n")
    f.write("def msgbox(*args, **kwargs):\n    return None\n")
    f.write("def buttonbox(*args, **kwargs):\n    return None\n")
    f.write("def ynbox(*args, **kwargs):\n    return True\n")
    f.write("def choicebox(*args, **kwargs):\n    return None\n")

print("v2e ready.")

# ============================================================
# RUN V2E
# ============================================================

os.makedirs(V2E_OUTPUT_DIR, exist_ok=True)

print(f"\nConverting {VIDEO_FILE} → events...")
print(f"Output: {V2E_OUTPUT_DIR}\n")

cmd = [
    "python", "/kaggle/working/v2e/v2e.py",
    "--input", VIDEO_FILE,
    "--output_folder", V2E_OUTPUT_DIR,
    "--dvs346",
    "--output_width", "346",
    "--output_height", "260",
    "--disable_slomo",
    "--dvs_h5", "events.h5",
    "--no_preview",
    "--overwrite"
]

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    h5_path = os.path.join(V2E_OUTPUT_DIR, "events.h5")
    if os.path.exists(h5_path):
        size_mb = os.path.getsize(h5_path) / (1024**2)
        print(f"\n✓ Success!")
        print(f"  H5 file: {h5_path}")
        print(f"  Size: {size_mb:.2f} MB")
        print(f"\nUse this in the pipeline:")
        print(f"  h5_file='{h5_path}'")
    else:
        print("✗ H5 file not created (v2e may have failed)")
else:
    print(f"✗ v2e failed with return code {result.returncode}")

Installing v2e...
v2e ready.

Converting /kaggle/input/datasets/virajsaws/meteor-dataset-allsky7/2026_05_15_01_29_01_000_012661_trim300_wm.mp4 → events...
Output: /kaggle/working/v2e_output



INFO:__main__:torch device is cuda
INFO:__main__:No module named 'gooey': Gooey GUI builder not available, will use command line arguments.
Install with 'pip install Gooey if you want a no-arg GUI to invoke v2e'. See README
INFO:__main__:name 'Gooey' is not defined: Gooey package GUI not available, using command line arguments. 
You can try to install with "pip install Gooey"
INFO:v2ecore.v2e_utils:using output folder /kaggle/working/v2e_output
INFO:__main__:output_in_place==False so made output_folder=/kaggle/working/v2e_output
INFO:v2ecore.v2e_args:
*** arguments:
auto_timestamp_resolution:	True
avi_frame_rate:	30
batch_size:	8
crop:	None
cs_lambda_pixels:	None
cs_tau_p_ms:	None
cutoff_hz:	300
ddd_output:	False
disable_slomo:	True
dvs1024:	False
dvs128:	False
dvs240:	False
dvs346:	True
dvs640:	False
dvs_aedat2:	None
dvs_aedat4:	None
dvs_emulator_seed:	0
dvs_exposure:	['duration', '0.01']
dvs_h5:	events.h5
dvs_params:	None
dvs_text:	None
dvs_vid:	dvs-video.avi
dvs_vid_full_scale:	2
hd

✗ v2e failed with return code 1


In [8]:
!pip install -q dv-processing evt3 h5py opencv-python-headless scikit-image pandas tqdm
print("Dependencies installed.")

Dependencies installed.


In [9]:
import os
import cv2
import h5py
import numpy as np
import pandas as pd
import logging
from pathlib import Path
from typing import List, Dict, Tuple, Any, Optional
from dataclasses import dataclass
from collections import deque
from tqdm import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
logger = logging.getLogger(__name__)

print("Imports complete.")

Imports complete.


# **Main logic**

In [10]:
@dataclass
class PipelineConfig:
    """All pipeline parameters in one place."""
    
    # I/O
    h5_file: str = "/kaggle/working/v2e_output/events.h5"  # SET THIS: path to HDF5 from v2e
    output_dir: str = "/kaggle/working/meteor_output"
    output_video: str = ""  # auto-generated
    output_csv: str = ""   # auto-generated
    
    # Temporal Windowing
    window_us: int = 100_000      # µs per window
    step_us: int = 100_000        # step size (no overlap if equal)
    
    # Blob Detection
    blob_min_area: int = 2
    blob_max_area: int = 500
    threshold_percentile: int = 90
    morph_kernel_size: int = 3
    
    # Tracking
    max_association_distance: float = 50.0  # pixels
    max_missed_windows: int = 3
    min_track_length: int = 3
    min_track_duration_s: float = 0.10
    
    # Scoring
    meteor_score_threshold: float = 0.80
    
    # Video Output
    video_fps: int = 10
    video_scale: int = 3
    
    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)
        if not self.output_video:
            self.output_video = os.path.join(self.output_dir, "meteor_tracks.mp4")
        if not self.output_csv:
            self.output_csv = os.path.join(self.output_dir, "meteor_tracks.csv")

# CHANGE THIS TO YOUR H5 FILE
config = PipelineConfig(
    h5_file="/kaggle/working/v2e_output/events.h5"
)

print(f"Config loaded.")
print(f"  H5: {config.h5_file}")
print(f"  Output dir: {config.output_dir}")

Config loaded.
  H5: /kaggle/working/v2e_output/events.h5
  Output dir: /kaggle/working/meteor_output


In [11]:
def load_events(h5_path: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, int, int]:
    """
    Load event data from HDF5.
    
    Returns:
        t (µs), x, y arrays and (width, height)
    """
    logger.info(f"Loading events from {h5_path}...")
    
    if not os.path.exists(h5_path):
        raise FileNotFoundError(f"H5 file not found: {h5_path}")
    
    with h5py.File(h5_path, 'r') as f:
        if 'events' not in f:
            raise KeyError(f"'events' dataset not found in {h5_path}")
        
        events = f['events'][:]
    
    # Expected format: [timestamp, x, y, polarity]
    if events.shape[1] != 4:
        raise ValueError(f"Expected 4 columns, got {events.shape[1]}")
    
    t = events[:, 0].astype(np.int64)
    x = events[:, 1].astype(np.int32)
    y = events[:, 2].astype(np.int32)
    # p = events[:, 3]  # unused for now
    
    width = int(x.max()) + 1
    height = int(y.max()) + 1
    
    logger.info(f"  Events: {len(t):,}")
    logger.info(f"  Resolution: {width} x {height}")
    logger.info(f"  Time range: {t.min():,} → {t.max():,} µs ({(t.max()-t.min())/1e6:.2f}s)")
    
    return t, x, y, width, height

# Load
t, x, y, width, height = load_events(config.h5_file)

2026-08-24 07:31:34,640 | INFO | Loading events from /kaggle/working/v2e_output/events.h5...
2026-08-24 07:31:34,652 | INFO |   Events: 46,920
2026-08-24 07:31:34,653 | INFO |   Resolution: 346 x 260
2026-08-24 07:31:34,654 | INFO |   Time range: 13,333 → 10,960,000 µs (10.95s)


In [12]:
def detect_blobs_in_window(
    t: np.ndarray,
    x: np.ndarray,
    y: np.ndarray,
    width: int,
    height: int,
    window_start: int,
    window_end: int,
    config: PipelineConfig
) -> List[Dict[str, Any]]:
    """
    Detect connected components (blobs) in a temporal window.
    """
    # Mask events in this window
    mask = (t >= window_start) & (t < window_end)
    ex, ey = x[mask], y[mask]
    
    if len(ex) == 0:
        return []
    
    # Count events per pixel
    count_img = np.zeros((height, width), dtype=np.uint16)
    np.add.at(count_img, (ey, ex), 1)
    
    if count_img.max() == 0:
        return []
    
    # Normalize to 0–255
    activity = cv2.normalize(count_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # Threshold at percentile
    nonzero = activity[activity > 0]
    if len(nonzero) == 0:
        return []
    
    threshold = max(5, int(np.percentile(nonzero, config.threshold_percentile)))
    binary = (activity >= threshold).astype(np.uint8) * 255
    
    # Morphology
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (config.morph_kernel_size, config.morph_kernel_size)
    )
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    
    # Connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )
    
    blobs = []
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        
        if not (config.blob_min_area <= area <= config.blob_max_area):
            continue
        
        cx, cy = centroids[i]
        component = (labels == i)
        event_count = int(count_img[component].sum())
        
        blobs.append({
            "cx": float(cx),
            "cy": float(cy),
            "x": int(stats[i, cv2.CC_STAT_LEFT]),
            "y": int(stats[i, cv2.CC_STAT_TOP]),
            "w": int(stats[i, cv2.CC_STAT_WIDTH]),
            "h": int(stats[i, cv2.CC_STAT_HEIGHT]),
            "area": int(area),
            "events": event_count
        })
    
    return blobs

# Quick test
test_blobs = detect_blobs_in_window(
    t, x, y, width, height,
    int(t.min()), int(t.min()) + config.window_us,
    config
)
logger.info(f"Test window blob detection: {len(test_blobs)} blobs found")

2026-08-24 07:31:34,699 | INFO | Test window blob detection: 22 blobs found


In [13]:
class MeteorTracker:
    """Track moving objects and score for meteor-likeness."""
    
    def __init__(self, config: PipelineConfig, width: int, height: int):
        self.config = config
        self.width = width
        self.height = height
        
        self.tracks: List[Dict[str, Any]] = []
        self.next_track_id = 0
        self.blob_cache: Dict[int, List[Dict]] = {}
    
    def create_track(self, blob: Dict, timestamp_s: float):
        """Start a new track."""
        track = {
            "id": self.next_track_id,
            "points": [(timestamp_s, blob["cx"], blob["cy"])],
            "blobs": [blob],
            "missed": 0,
            "active": True
        }
        self.tracks.append(track)
        self.next_track_id += 1
    
    def process_windows(
        self,
        t: np.ndarray,
        x: np.ndarray,
        y: np.ndarray,
    ):
        """Process all temporal windows."""
        start_time = int(t.min())
        end_time = int(t.max())
        
        window_times = np.arange(
            start_time,
            end_time + config.window_us,
            config.step_us
        )
        
        logger.info(f"Processing {len(window_times)} windows...")
        
        for window_start in tqdm(window_times, desc="Tracking"):
            window_end = window_start + config.window_us
            blobs = detect_blobs_in_window(
                t, x, y, self.width, self.height,
                window_start, window_end,
                config
            )
            
            # Cache for visualization
            self.blob_cache[window_start] = blobs
            
            timestamp_s = (window_start - start_time) / 1e6
            active_tracks = [tr for tr in self.tracks if tr["active"]]
            assignments = set()
            
            # Try to associate blobs to existing tracks
            for track in active_tracks:
                last_t, last_x, last_y = track["points"][-1]
                candidates = []
                
                for blob_idx, blob in enumerate(blobs):
                    if blob_idx in assignments:
                        continue
                    
                    dx = blob["cx"] - last_x
                    dy = blob["cy"] - last_y
                    dist = np.sqrt(dx**2 + dy**2)
                    
                    if dist <= config.max_association_distance:
                        candidates.append((dist, blob_idx, blob))
                
                if candidates:
                    candidates.sort(key=lambda z: z[0])
                    _, blob_idx, blob = candidates[0]
                    
                    assignments.add(blob_idx)
                    track["points"].append((timestamp_s, blob["cx"], blob["cy"]))
                    track["blobs"].append(blob)
                    track["missed"] = 0
                else:
                    track["missed"] += 1
                    if track["missed"] > config.max_missed_windows:
                        track["active"] = False
            
            # Create new tracks for unassigned blobs
            for blob_idx, blob in enumerate(blobs):
                if blob_idx not in assignments:
                    self.create_track(blob, timestamp_s)
    
    def score_track(self, track: Dict) -> Optional[Dict[str, float]]:
        """Score a trajectory for meteor-likeness."""
        points = track["points"]
        
        if len(points) < 3:
            return None
        
        times = np.array([p[0] for p in points])
        xs = np.array([p[1] for p in points])
        ys = np.array([p[2] for p in points])
        
        duration = times[-1] - times[0]
        if duration <= 0:
            return None
        
        # Path geometry
        dx = np.diff(xs)
        dy = np.diff(ys)
        step_dist = np.sqrt(dx**2 + dy**2)
        path_length = step_dist.sum()
        displacement = np.sqrt((xs[-1] - xs[0])**2 + (ys[-1] - ys[0])**2)
        
        if path_length <= 0:
            return None
        
        # Straightness
        straightness = displacement / path_length
        
        # Velocity
        velocity = displacement / duration
        
        # Direction consistency
        angles = np.arctan2(dy, dx)
        overall_angle = np.arctan2(ys[-1] - ys[0], xs[-1] - xs[0])
        angle_diff = np.arctan2(
            np.sin(angles - overall_angle),
            np.cos(angles - overall_angle)
        )
        mean_dir_error = np.mean(np.abs(np.degrees(angle_diff)))
        
        # Step consistency
        if len(step_dist) > 1:
            mean_step = np.mean(step_dist)
            std_step = np.std(step_dist)
            step_consistency = mean_step / (mean_step + std_step + 1e-6)
        else:
            step_consistency = 0.0
        
        # Sub-scores (0–1 each)
        persistence_score = min(len(points) / 8.0, 1.0)
        duration_score = min(duration / 1.0, 1.0)
        displacement_score = min(displacement / 30.0, 1.0)
        straightness_score = np.clip(straightness, 0, 1)
        direction_score = np.clip(1.0 - (mean_dir_error / 90.0), 0, 1)
        step_score = np.clip(step_consistency, 0, 1)
        
        # Weighted final score
        path_score = (
            0.25 * persistence_score +
            0.15 * duration_score +
            0.15 * displacement_score +
            0.25 * straightness_score +
            0.10 * direction_score +
            0.10 * step_score
        )
        
        return {
            "duration": duration,
            "points": len(points),
            "path_length": path_length,
            "displacement": displacement,
            "velocity": velocity,
            "straightness": straightness,
            "direction_error": mean_dir_error,
            "step_consistency": step_consistency,
            "path_score": path_score
        }
    
    def filter_and_rank(self) -> Tuple[List, List]:
        """Filter tracks and return (all_valid, meteor_candidates)."""
        all_results = []
        
        for track in self.tracks:
            score = self.score_track(track)
            if not score:
                continue
            if score["points"] < config.min_track_length:
                continue
            if score["duration"] < config.min_track_duration_s:
                continue
            
            all_results.append((track, score))
        
        all_results.sort(key=lambda z: z[1]["path_score"], reverse=True)
        
        meteor_tracks = [
            (t, r) for t, r in all_results
            if r["path_score"] >= config.meteor_score_threshold
        ]
        
        logger.info(f"Raw tracks: {len(self.tracks)}")
        logger.info(f"Valid tracks: {len(all_results)}")
        logger.info(f"Meteor candidates (score ≥ {config.meteor_score_threshold}): {len(meteor_tracks)}")
        
        return all_results, meteor_tracks

# Initialize
tracker = MeteorTracker(config, width, height)
logger.info("Tracker initialized.")

2026-08-24 07:31:34,785 | INFO | Tracker initialized.


In [14]:
# Process all windows
tracker.process_windows(t, x, y)

# Score and filter
all_tracks, meteor_tracks = tracker.filter_and_rank()

# Print top 10
logger.info("\n=== TOP 10 TRACKS ===")
for rank, (track, score) in enumerate(all_tracks[:10], 1):
    is_meteor = "✓ METEOR" if score["path_score"] >= config.meteor_score_threshold else ""
    print(
        f"#{rank} Track {track['id']} | Score {score['path_score']:.2f} {is_meteor}"
        f"\n  Duration: {score['duration']:.2f}s | Points: {score['points']}"
        f"\n  Velocity: {score['velocity']:.1f}px/s | Straightness: {score['straightness']:.2f}"
        f"\n  Direction error: {score['direction_error']:.1f}° | Step consistency: {score['step_consistency']:.2f}"
        f"\n"
    )

2026-08-24 07:31:34,806 | INFO | Processing 111 windows...
Tracking: 100%|██████████| 111/111 [00:00<00:00, 567.67it/s]
2026-08-24 07:31:35,016 | INFO | Raw tracks: 105
2026-08-24 07:31:35,017 | INFO | Valid tracks: 13
2026-08-24 07:31:35,018 | INFO | Meteor candidates (score ≥ 0.8): 2
2026-08-24 07:31:35,018 | INFO | 
=== TOP 10 TRACKS ===


#1 Track 32 | Score 0.94 ✓ METEOR
  Duration: 5.70s | Points: 50
  Velocity: 28.3px/s | Straightness: 0.96
  Direction error: 13.4° | Step consistency: 0.63

#2 Track 52 | Score 0.85 ✓ METEOR
  Duration: 1.60s | Points: 9
  Velocity: 24.9px/s | Straightness: 0.74
  Direction error: 33.9° | Step consistency: 0.52

#3 Track 99 | Score 0.76 
  Duration: 0.50s | Points: 3
  Velocity: 104.0px/s | Straightness: 1.00
  Direction error: 0.0° | Step consistency: 0.93

#4 Track 83 | Score 0.75 
  Duration: 0.70s | Points: 3
  Velocity: 67.3px/s | Straightness: 1.00
  Direction error: 0.2° | Step consistency: 0.53

#5 Track 27 | Score 0.71 
  Duration: 1.30s | Points: 8
  Velocity: 39.7px/s | Straightness: 0.28
  Direction error: 76.0° | Step consistency: 0.71

#6 Track 4 | Score 0.64 
  Duration: 0.40s | Points: 3
  Velocity: 95.8px/s | Straightness: 0.83
  Direction error: 36.7° | Step consistency: 0.72

#7 Track 31 | Score 0.60 
  Duration: 0.70s | Points: 3
  Velocity: 43.7px/s | Straightness

In [15]:
def export_csv(
    tracker: MeteorTracker,
    all_tracks: List,
    meteor_score_threshold: float,
    output_csv: str
):
    """Export tracks to CSV."""
    rows = []
    
    for track, score in all_tracks:
        is_meteor = score["path_score"] >= meteor_score_threshold
        
        for point in track["points"]:
            rows.append({
                "track_id": track["id"],
                "time_s": point[0],
                "x": point[1],
                "y": point[2],
                "duration_s": score["duration"],
                "displacement_px": score["displacement"],
                "velocity_px_s": score["velocity"],
                "straightness": score["straightness"],
                "direction_error_deg": score["direction_error"],
                "step_consistency": score["step_consistency"],
                "path_score": score["path_score"],
                "is_meteor": is_meteor
            })
    
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    logger.info(f"CSV exported: {output_csv}")
    logger.info(f"  Total rows: {len(df)}")
    logger.info(f"  Unique tracks: {df['track_id'].nunique()}")
    logger.info(f"  Meteor tracks: {df[df['is_meteor']]['track_id'].nunique()}")

export_csv(tracker, all_tracks, config.meteor_score_threshold, config.output_csv)

2026-08-24 07:31:35,042 | INFO | CSV exported: /kaggle/working/meteor_output/meteor_tracks.csv
2026-08-24 07:31:35,042 | INFO |   Total rows: 105
2026-08-24 07:31:35,052 | INFO |   Unique tracks: 13
2026-08-24 07:31:35,056 | INFO |   Meteor tracks: 2


In [16]:
def create_visualization_video(
    tracker: MeteorTracker,
    meteor_tracks: List,
    t: np.ndarray,
    config: PipelineConfig
):
    """Generate MP4 with tracked paths overlaid."""
    logger.info("Generating video...")
    
    start_time = int(t.min())
    end_time = int(t.max())
    
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(
        config.output_video,
        fourcc,
        config.video_fps,
        (config.video_scale * tracker.width, config.video_scale * tracker.height)
    )
    
    window_times = np.arange(
        start_time,
        end_time + config.window_us,
        config.step_us
    )
    
    for window_start in tqdm(window_times, desc="Video"):
        elapsed = (window_start - start_time) / 1e6
        
        # Black frame
        frame = np.zeros((tracker.height, tracker.width, 3), dtype=np.uint8)
        
        # Draw all blobs (faint gray)
        blobs = tracker.blob_cache.get(window_start, [])
        for blob in blobs:
            cv2.circle(
                frame,
                (int(blob["cx"]), int(blob["cy"])),
                2,
                (50, 50, 50),
                -1
            )
        
        # Draw meteor trajectories (white)
        for track, score in meteor_tracks:
            visible = [p for p in track["points"] if p[0] <= elapsed]
            if len(visible) < 2:
                continue
            
            coords = [(int(p[1]), int(p[2])) for p in visible]
            
            for j in range(1, len(coords)):
                cv2.line(frame, coords[j-1], coords[j], (255, 255, 255), 2)
            
            # Current position
            cx, cy = coords[-1]
            cv2.circle(frame, (cx, cy), 4, (255, 255, 255), -1)
            cv2.putText(
                frame,
                f"S={score['path_score']:.2f}",
                (max(2, cx + 6), max(15, cy - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.32,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )
        
        # HUD
        cv2.rectangle(frame, (0, 0), (tracker.width, 30), (0, 0, 0), -1)
        cv2.putText(
            frame,
            f"T={elapsed:.2f}s | METEORS={len(meteor_tracks)}",
            (5, 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.40,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
        
        # Scale and write
        frame = cv2.resize(
            frame,
            (config.video_scale * tracker.width, config.video_scale * tracker.height),
            interpolation=cv2.INTER_NEAREST
        )
        writer.write(frame)
    
    writer.release()
    logger.info(f"Video saved: {config.output_video}")

create_visualization_video(tracker, meteor_tracks, t, config)

2026-08-24 07:31:35,068 | INFO | Generating video...
Video: 100%|██████████| 111/111 [00:00<00:00, 180.38it/s]
2026-08-24 07:31:35,689 | INFO | Video saved: /kaggle/working/meteor_output/meteor_tracks.mp4


In [17]:
from IPython.display import Video, display

logger.info(f"\n{'='*70}")
logger.info("PIPELINE COMPLETE")
logger.info(f"{'='*70}")
logger.info(f"Meteors detected: {len(meteor_tracks)}")
logger.info(f"Output directory: {config.output_dir}")
logger.info(f"Video: {config.output_video}")
logger.info(f"CSV: {config.output_csv}")

if len(meteor_tracks) > 0:
    logger.info("\nPlayback:")
    display(Video(config.output_video, embed=True))
else:
    logger.warning("No meteors detected above threshold.")

2026-08-24 07:31:35,696 | INFO | 
2026-08-24 07:31:35,698 | INFO | PIPELINE COMPLETE
2026-08-24 07:31:35,699 | INFO | ======================================================================
2026-08-24 07:31:35,700 | INFO | Meteors detected: 2
2026-08-24 07:31:35,701 | INFO | Output directory: /kaggle/working/meteor_output
2026-08-24 07:31:35,702 | INFO | Video: /kaggle/working/meteor_output/meteor_tracks.mp4
2026-08-24 07:31:35,703 | INFO | CSV: /kaggle/working/meteor_output/meteor_tracks.csv
2026-08-24 07:31:35,703 | INFO | 
Playback:


In [18]:
import pandas as pd

# Load the CSV you just generated
tracks_csv = "/kaggle/working/meteor_output/meteor_tracks.csv"
df = pd.read_csv(tracks_csv)

print("="*70)
print("METEOR DETECTION RESULTS")
print("="*70)

# Filter to meteor tracks only
meteor_df = df[df['is_meteor']]

print(f"\nTotal detections: {len(df)}")
print(f"Unique tracks: {df['track_id'].nunique()}")
print(f"Meteor tracks (score >= 0.75): {meteor_df['track_id'].nunique()}")

print("\n" + "="*70)
print("METEOR CANDIDATES")
print("="*70)

for track_id in sorted(meteor_df['track_id'].unique()):
    track = meteor_df[meteor_df['track_id'] == track_id].iloc[0]
    print(f"\nTrack {track_id}:")
    print(f"  Score: {track['path_score']:.2f}")
    print(f"  Duration: {track['duration_s']:.2f}s")
    print(f"  Displacement: {track['displacement_px']:.1f}px")
    print(f"  Velocity: {track['velocity_px_s']:.1f}px/s")
    print(f"  Straightness: {track['straightness']:.2f}")
    print(f"  Direction error: {track['direction_error_deg']:.1f}°")

METEOR DETECTION RESULTS

Total detections: 105
Unique tracks: 13
Meteor tracks (score >= 0.75): 2

METEOR CANDIDATES

Track 32:
  Score: 0.94
  Duration: 5.70s
  Displacement: 161.2px
  Velocity: 28.3px/s
  Straightness: 0.96
  Direction error: 13.4°

Track 52:
  Score: 0.85
  Duration: 1.60s
  Displacement: 39.8px
  Velocity: 24.9px/s
  Straightness: 0.74
  Direction error: 33.9°


# **Validation**

In [19]:
import cv2
import numpy as np
from IPython.display import Video, display

# CHANGE THIS to your original video file
ORIGINAL_VIDEO = "/kaggle/input/datasets/virajsaws/meteor-dataset-allsky7/2026_05_15_01_29_01_000_012661_trim300_wm.mp4"

OUTPUT_VIDEO = "/kaggle/working/meteor_output/validation_overlay.mp4"

# Load tracks
meteor_df = df[df['is_meteor']]
meteor_tracks = {}
for track_id in meteor_df['track_id'].unique():
    track_data = meteor_df[meteor_df['track_id'] == track_id].sort_values('time_s')
    meteor_tracks[track_id] = track_data

# Open original video
cap = cv2.VideoCapture(ORIGINAL_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Original video: {width}x{height} @ {fps}fps, {total_frames} frames")

# Scale factor (original is ~1920x1080, events are 346x260)
scale_x = width / 346
scale_y = height / 260

print(f"Scale factor: x={scale_x:.2f}, y={scale_y:.2f}")

# Writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Time in seconds
    time_s = frame_idx / fps
    
    # Draw all meteor tracks up to this time
    for track_id, track_data in meteor_tracks.items():
        # Get points visible at this time
        visible = track_data[track_data['time_s'] <= time_s]
        
        if len(visible) < 1:
            continue
        
        # Convert event coords back to original video coords
        coords = [
            (int(row['x'] * scale_x), int(row['y'] * scale_y))
            for _, row in visible.iterrows()
        ]
        
        # Draw trajectory line
        for i in range(1, len(coords)):
            cv2.line(frame, coords[i-1], coords[i], (0, 255, 255), 2)
        
        # Draw current point
        if len(coords) > 0:
            cx, cy = coords[-1]
            cv2.circle(frame, (cx, cy), 8, (0, 255, 255), -1)
            score = visible.iloc[-1]['path_score']
            cv2.putText(
                frame,
                f"T{track_id} S={score:.2f}",
                (cx + 10, cy - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 255),
                2
            )
    
    # HUD
    cv2.rectangle(frame, (0, 0), (width, 40), (0, 0, 0), -1)
    cv2.putText(
        frame,
        f"T={time_s:.2f}s | Meteors={len([t for t in meteor_tracks if meteor_df[meteor_df['track_id']==t]['time_s'].max() >= time_s])}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2
    )
    
    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print(f"✓ Overlay video saved: {OUTPUT_VIDEO}")
print("Play it to see if the yellow circles match the actual meteors in the video.")

display(Video(OUTPUT_VIDEO, embed=True))

Original video: 1920x1080 @ 25.0fps, 275 frames
Scale factor: x=5.55, y=4.15
✓ Overlay video saved: /kaggle/working/meteor_output/validation_overlay.mp4
Play it to see if the yellow circles match the actual meteors in the video.


# **Raw data visualization**

In [20]:
# import matplotlib.pyplot as plt
# import numpy as np
# from matplotlib.colors import LogNorm

# # Pick one window to inspect
# sample_window_start = int(t.min()) + config.window_us * 30  # window 30
# sample_window_end = sample_window_start + config.window_us

# # ============================================================
# # RAW EVENTS (ALL NOISE + SIGNAL)
# # ============================================================

# mask_raw = (t >= sample_window_start) & (t < sample_window_end)
# ex_raw = x[mask_raw]
# ey_raw = y[mask_raw]

# # Create 2D histogram of raw events
# raw_histogram = np.zeros((height, width), dtype=np.uint32)
# np.add.at(raw_histogram, (ey_raw, ex_raw), 1)

# # ============================================================
# # CLEANED EVENTS (THRESHOLDED)
# # ============================================================

# # Normalize
# activity = cv2.normalize(raw_histogram.astype(np.float32), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# # Threshold at percentile
# nonzero = activity[activity > 0]
# threshold_val = max(5, int(np.percentile(nonzero, config.threshold_percentile)))
# binary = (activity >= threshold_val).astype(np.uint8) * 255

# # Morphology
# kernel = cv2.getStructuringElement(
#     cv2.MORPH_ELLIPSE,
#     (config.morph_kernel_size, config.morph_kernel_size)
# )
# binary_clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
# binary_clean = cv2.morphologyEx(binary_clean, cv2.MORPH_CLOSE, kernel)

# # ============================================================
# # VISUALIZATION
# # ============================================================

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# # Plot 1: Raw events (scatter)
# ax = axes[0]
# ax.scatter(ex_raw, ey_raw, s=1, alpha=0.3, c='blue')
# ax.set_xlim(0, width)
# ax.set_ylim(height, 0)  # flip Y
# ax.set_title(f'RAW EVENTS\n{len(ex_raw)} events (all noise + signal)', fontsize=12, fontweight='bold')
# ax.set_xlabel('X (pixels)')
# ax.set_ylabel('Y (pixels)')
# ax.grid(True, alpha=0.3)

# # Plot 2: Raw as heatmap
# ax = axes[1]
# im = ax.imshow(raw_histogram, cmap='hot', norm=LogNorm(vmin=1, vmax=raw_histogram.max()))
# ax.set_title(f'RAW HEATMAP (log scale)\nThreshold={threshold_val} @ {config.threshold_percentile}th percentile', 
#              fontsize=12, fontweight='bold')
# ax.set_xlabel('X (pixels)')
# ax.set_ylabel('Y (pixels)')
# plt.colorbar(im, ax=ax, label='Event count')

# # Plot 3: Cleaned (binary)
# ax = axes[2]
# ax.imshow(binary_clean, cmap='gray')
# ax.set_title(f'CLEANED + DETECTED BLOBS\nAfter threshold + morphology', 
#              fontsize=12, fontweight='bold')
# ax.set_xlabel('X (pixels)')
# ax.set_ylabel('Y (pixels)')

# plt.tight_layout()
# plt.savefig('/kaggle/working/meteor_output/raw_vs_cleaned.png', dpi=100, bbox_inches='tight')
# print("✓ Saved: raw_vs_cleaned.png")
# plt.show()

# # ============================================================
# # STATISTICS
# # ============================================================

# print(f"\n{'='*70}")
# print(f"NOISE CLEANING ANALYSIS (Window {sample_window_start:,} µs)")
# print(f"{'='*70}")
# print(f"\nRAW EVENTS:")
# print(f"  Total events: {len(ex_raw):,}")
# print(f"  Unique pixels: {len(np.unique(np.column_stack((ex_raw, ey_raw)), axis=0))}")
# print(f"  Event density: {len(ex_raw)/(width*height):.2f} events/pixel (avg)")
# print(f"  Max events in any pixel: {raw_histogram.max()}")
# print(f"  Median events per active pixel: {np.median(raw_histogram[raw_histogram>0]):.1f}")

# print(f"\nTHRESHOLD:")
# print(f"  90th percentile of activity: {threshold_val}")
# print(f"  Only keeping pixels with ≥{threshold_val} events")

# print(f"\nAFTER THRESHOLD:")
# print(f"  Pixels above threshold: {np.sum(activity >= threshold_val)}")
# print(f"  Reduction: {100*(1 - np.sum(activity >= threshold_val)/(width*height)):.1f}% of pixels removed")

# print(f"\nAFTER MORPHOLOGY:")
# num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_clean, connectivity=8)
# valid_blobs = 0
# for i in range(1, num_labels):
#     area = stats[i, cv2.CC_STAT_AREA]
#     if config.blob_min_area <= area <= config.blob_max_area:
#         valid_blobs += 1

# print(f"  Total connected components: {num_labels - 1}")
# print(f"  Valid blobs (size {config.blob_min_area}-{config.blob_max_area}px): {valid_blobs}")
# print(f"  Noise rejection rate: {100*(1 - valid_blobs/max(1,num_labels-1)):.1f}%")

In [21]:
# # Same window, now show the trajectory overlaid

# fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# # Get all events in this window for scatter
# mask = (t >= sample_window_start) & (t < sample_window_end)
# ex, ey = x[mask], y[mask]

# # Plot 1: Raw scatter
# ax = axes[0, 0]
# ax.scatter(ex, ey, s=1, alpha=0.2, c='blue', label='Raw events')
# ax.set_xlim(0, width)
# ax.set_ylim(height, 0)
# ax.set_title('1. Raw Events (Noisy)', fontweight='bold')
# ax.set_xlabel('X')
# ax.set_ylabel('Y')
# ax.grid(True, alpha=0.2)
# ax.legend()

# # Plot 2: Cleaned blobs
# ax = axes[0, 1]
# ax.imshow(binary_clean, cmap='gray', extent=[0, width, height, 0])
# ax.set_title('2. Cleaned (Thresholded + Morphology)', fontweight='bold')
# ax.set_xlabel('X')
# ax.set_ylabel('Y')

# # Overlay blob centroids
# num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_clean, connectivity=8)
# for i in range(1, num_labels):
#     area = stats[i, cv2.CC_STAT_AREA]
#     if config.blob_min_area <= area <= config.blob_max_area:
#         cx, cy = centroids[i]
#         ax.plot(cx, cy, 'r+', markersize=15, markeredgewidth=2)

# # Plot 3: Heatmap with threshold line
# ax = axes[1, 0]
# activity = cv2.normalize(raw_histogram, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
# im = ax.imshow(activity, cmap='hot', extent=[0, width, height, 0])
# ax.contour(activity, levels=[threshold_val], colors='cyan', linewidths=2)
# ax.set_title(f'3. Activity Heatmap (Cyan=threshold at {threshold_val})', fontweight='bold')
# ax.set_xlabel('X')
# ax.set_ylabel('Y')
# plt.colorbar(im, ax=ax)

# # Plot 4: Detected blobs (circles)
# ax = axes[1, 1]
# frame_viz = np.zeros((height, width, 3), dtype=np.uint8)
# frame_viz[:, :, 0] = activity  # Red channel = activity

# for i in range(1, num_labels):
#     area = stats[i, cv2.CC_STAT_AREA]
#     if config.blob_min_area <= area <= config.blob_max_area:
#         cx, cy = centroids[i]
#         cv2.circle(frame_viz, (int(cx), int(cy)), 10, (0, 255, 0), 2)
#         cv2.putText(frame_viz, f'{i}', (int(cx)-5, int(cy)-15), 
#                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

# ax.imshow(frame_viz)
# ax.set_title(f'4. Detected Blobs (n={valid_blobs})', fontweight='bold')
# ax.set_xlabel('X')
# ax.set_ylabel('Y')

# plt.tight_layout()
# plt.savefig('/kaggle/working/meteor_output/pipeline_stages.png', dpi=100, bbox_inches='tight')
# print("✓ Saved: pipeline_stages.png")
# plt.show()

In [22]:
# # Find a meteor track and a noise track, compare them side-by-side

# meteor_df = df[df['is_meteor']]
# non_meteor_df = df[~df['is_meteor']]

# if len(meteor_df) > 0 and len(non_meteor_df) > 0:
#     meteor_track_id = meteor_df['track_id'].iloc[0]
#     noise_track_id = non_meteor_df['track_id'].iloc[0]
    
#     meteor_track = df[df['track_id'] == meteor_track_id]
#     noise_track = df[df['track_id'] == noise_track_id]
    
#     print(f"{'='*70}")
#     print(f"METEOR vs NOISE COMPARISON")
#     print(f"{'='*70}")
    
#     print(f"\n🎯 METEOR (Track {meteor_track_id}):")
#     print(f"  Score: {meteor_track.iloc[0]['path_score']:.2f}")
#     print(f"  Duration: {meteor_track.iloc[0]['duration_s']:.2f}s")
#     print(f"  Points: {len(meteor_track)}")
#     print(f"  Straightness: {meteor_track.iloc[0]['straightness']:.2f}")
#     print(f"  Direction error: {meteor_track.iloc[0]['direction_error_deg']:.1f}°")
#     print(f"  Step consistency: {meteor_track.iloc[0]['step_consistency']:.2f}")
#     print(f"  Velocity: {meteor_track.iloc[0]['velocity_px_s']:.1f}px/s")
    
#     print(f"\n💥 NOISE (Track {noise_track_id}):")
#     print(f"  Score: {noise_track.iloc[0]['path_score']:.2f}")
#     print(f"  Duration: {noise_track.iloc[0]['duration_s']:.2f}s")
#     print(f"  Points: {len(noise_track)}")
#     print(f"  Straightness: {noise_track.iloc[0]['straightness']:.2f}")
#     print(f"  Direction error: {noise_track.iloc[0]['direction_error_deg']:.1f}°")
#     print(f"  Step consistency: {noise_track.iloc[0]['step_consistency']:.2f}")
#     print(f"  Velocity: {noise_track.iloc[0]['velocity_px_s']:.1f}px/s")
    
#     # Plot trajectories
#     fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
#     # Meteor
#     ax = axes[0]
#     ax.plot(meteor_track['x'], meteor_track['y'], 'yo-', linewidth=2, markersize=6)
#     ax.set_xlim(0, width)
#     ax.set_ylim(height, 0)
#     ax.grid(True, alpha=0.3)
#     ax.set_title(f'🎯 METEOR (Score {meteor_track.iloc[0]["path_score"]:.2f})\nStraight, consistent, fast', 
#                 fontweight='bold', fontsize=12)
#     ax.set_xlabel('X (pixels)')
#     ax.set_ylabel('Y (pixels)')
    
#     # Noise
#     ax = axes[1]
#     ax.plot(noise_track['x'], noise_track['y'], 'r+--', linewidth=1, markersize=8, markeredgewidth=2)
#     ax.set_xlim(0, width)
#     ax.set_ylim(height, 0)
#     ax.grid(True, alpha=0.3)
#     ax.set_title(f'💥 NOISE (Score {noise_track.iloc[0]["path_score"]:.2f})\nErratic, jumpy, short', 
#                 fontweight='bold', fontsize=12)
#     ax.set_xlabel('X (pixels)')
#     ax.set_ylabel('Y (pixels)')
    
#     plt.tight_layout()
#     plt.savefig('/kaggle/working/meteor_output/meteor_vs_noise.png', dpi=100, bbox_inches='tight')
#     print("\n✓ Saved: meteor_vs_noise.png")
#     plt.show()
# else:
#     print("No meteors or noise tracks to compare")